# Simple linear BRDF — reflectance / radiance vs SZA and cos(SZA)

For a single **patch** and a single **wavelength**, this notebook pulls every
QC-passed acquisition of that patch and plots four scatter panels:

| | vs SZA | vs cos(SZA) |
|---|---|---|
| **Reflectance** | panel 1 | panel 3 |
| **Radiance**    | panel 2 | panel 4 |

Each panel gets **two** regression lines — ordinary least squares (OLS) and the
robust Theil–Sen slope (the repo's drift-analysis convention) — with the fit
statistics printed in the top-right corner.

**Why cos(SZA)?** In the whiteboard BRDF model (see `brdf_exploration.ipynb`) the
surface directional response is the linear *bracket* $c + d\cos\theta$, so a
straight line through reflectance-vs-$\cos\theta$ recovers the site's BRDF
sensitivity ($d/c$ = slope/intercept). Radiance carries the same geometry factor
times the raw solar/instrument signal, shown alongside for reference.

**One point per acquisition.** SZA is essentially constant across a 10 km patch
within a single overpass, so the regression signal lives *across scenes*: each
point is the patch-**median** reflectance/radiance at the chosen wavelength for
one acquisition, plotted against that scene's mean SZA. Points are colored by
acquisition year, so any time/geometry confounding is visible by eye.

**Inputs consumed** (all written by the extraction step; no network here):
`data/geometry/geometry_{PICS}_{PATCH}.csv`,
`data/reflectance/{PICS}_{PATCH}_{ts}_perchannel_rfl.csv`,
`data/radiance/{PICS}_{PATCH}_{ts}_perchannel_rad.csv`, and the QC worksheet
`data/qc/{PICS}_{PATCH}_qc.csv`.


## 1. Configuration

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# ------------------------------------------------------------------
# USER-CONFIGURABLE INPUTS
# ------------------------------------------------------------------
PICS_ID       = "Libya_4"     # underscored site id
PATCH_ID      = "10_7_1"      # patch_id within the PICS
WAVELENGTH_NM = 660.0         # target wavelength (nm); nearest EMIT channel is used

Y_STAT   = "median"           # patch statistic for the y-value: median | mean | ...
SZA_STAT = "sza_mean"         # per-scene SZA column from the geometry table
APPLY_QC = True               # drop keep=FALSE scenes (falls back to all if no sheet)

# ------------------------------------------------------------------
# Repo-relative paths (walk up until we find the package dir).
# ------------------------------------------------------------------
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "emit_pics").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

DATA_ROOT = REPO_ROOT / "data"
GEOM_PATH = DATA_ROOT / "geometry" / f"geometry_{PICS_ID}_{PATCH_ID}.csv"
RFL_DIR   = DATA_ROOT / "reflectance"
RAD_DIR   = DATA_ROOT / "radiance"
QC_PATH   = DATA_ROOT / "qc" / f"{PICS_ID}_{PATCH_ID}_qc.csv"
FIG_DIR   = REPO_ROOT / "figures" / "BRDF"

RFL_LABEL = "Reflectance (unitless)"
RAD_LABEL = "Radiance (µW cm⁻² sr⁻¹ nm⁻¹)"

plt.rcParams["figure.dpi"] = 110
print(f"Repo root:  {REPO_ROOT}")
print(f"Patch:      {PICS_ID} / {PATCH_ID}   @ {WAVELENGTH_NM:.1f} nm ({Y_STAT})")
print(f"Geometry:   {GEOM_PATH}   exists={GEOM_PATH.exists()}")

## 2. Load the scenes and apply the QC gate

Scenes come from the per-scene geometry table. `apply_qc_filter` is called with
`require_review=False`, so it drops any `keep=FALSE` scenes if a reviewed
worksheet exists and otherwise passes every scene through — it never blocks the
notebook.

In [ ]:
from emit_pics.qc_gate import apply_qc_filter

if not GEOM_PATH.exists():
    raise FileNotFoundError(
        f"Geometry table not found:\n  {GEOM_PATH}\n"
        "Run the extraction step (run_single_patch_cube_extract.py or "
        "run_PICS_allpatches_cube_extract.py) for this patch first."
    )

scenes = pd.read_csv(GEOM_PATH).sort_values("datetime").reset_index(drop=True)
n_all = len(scenes)

if APPLY_QC:
    scenes, qc_report = apply_qc_filter(scenes, QC_PATH, require_review=False,
                                        timestamp_col="timestamp")
    print(f"QC: {qc_report['n_before']} -> {qc_report['n_after']} scenes "
          f"({qc_report['n_excluded']} excluded {qc_report['reasons']})")
    if not QC_PATH.exists():
        print(f"  (no QC worksheet at {QC_PATH.name}; using all scenes)")
else:
    print("QC filtering disabled (APPLY_QC=False).")

print(f"{len(scenes)} of {n_all} scene(s) will be plotted.")
scenes[["timestamp", "datetime", "has_rfl", SZA_STAT]].head()

## 3. Assemble one (SZA, reflectance, radiance) point per scene

For each surviving scene we read its per-channel reflectance and radiance CSVs
and pick the channel nearest the target wavelength (reflectance and radiance have
slightly different band grids, so each is matched independently). RAD-only scenes
contribute a radiance point with reflectance left as NaN.

In [ ]:
def nearest_channel(csv_path: Path, target_nm: float, stat_col: str):
    """(actual_wl, value) at the channel nearest *target_nm*, or (nan, nan)."""
    if not csv_path.exists():
        return np.nan, np.nan
    d = pd.read_csv(csv_path)
    if stat_col not in d.columns or "central_wl" not in d.columns:
        return np.nan, np.nan
    i = (d["central_wl"] - target_nm).abs().idxmin()
    return float(d["central_wl"].iloc[i]), float(d[stat_col].iloc[i])


rows = []
rfl_wls, rad_wls = [], []
for _, s in scenes.iterrows():
    ts = str(s["timestamp"])
    rfl_wl, rfl = nearest_channel(RFL_DIR / f"{PICS_ID}_{PATCH_ID}_{ts}_perchannel_rfl.csv",
                                  WAVELENGTH_NM, f"{Y_STAT}_rfl")
    rad_wl, rad = nearest_channel(RAD_DIR / f"{PICS_ID}_{PATCH_ID}_{ts}_perchannel_rad.csv",
                                  WAVELENGTH_NM, f"{Y_STAT}_rad")
    if np.isfinite(rfl_wl): rfl_wls.append(rfl_wl)
    if np.isfinite(rad_wl): rad_wls.append(rad_wl)
    rows.append({"timestamp": ts, "datetime": s["datetime"],
                 "sza": float(s[SZA_STAT]), "rfl": rfl, "rad": rad})

df = pd.DataFrame(rows)
df["cos_sza"] = np.cos(np.radians(df["sza"]))
_dt = pd.to_datetime(df["datetime"], utc=True, errors="coerce")
df["year"] = _dt.dt.year + (_dt.dt.dayofyear - 1) / 365.25

# Report the actual channels used (should be one value each).
rfl_ch = f"{np.median(rfl_wls):.1f} nm" if rfl_wls else "n/a"
rad_ch = f"{np.median(rad_wls):.1f} nm" if rad_wls else "n/a"
print(f"Target {WAVELENGTH_NM:.1f} nm  ->  reflectance channel {rfl_ch}, radiance channel {rad_ch}")
print(f"{df['rfl'].notna().sum()} reflectance point(s), {df['rad'].notna().sum()} radiance point(s), "
      f"SZA range {df['sza'].min():.1f}–{df['sza'].max():.1f}°")
df

## 4. Regression helper (OLS + Theil–Sen)

`fit_stats` returns both fits on the finite (x, y) pairs. `plot_panel` draws the
scatter (colored by year), overlays both regression lines, and annotates the
top-right corner with slope / R² / p for OLS and slope / Kendall-τ / p for
Theil–Sen, plus the point count.

In [ ]:
def fit_stats(x, y):
    """OLS + Theil–Sen fit over finite pairs. Returns a dict (n always present)."""
    x = np.asarray(x, float); y = np.asarray(y, float)
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]
    out = {"n": int(x.size)}
    if x.size < 3 or np.ptp(x) == 0:
        return out                         # too few points / no x-spread to fit
    ols = stats.linregress(x, y)
    ts = stats.theilslopes(y, x)           # (slope, intercept, lo_slope, hi_slope)
    tau, tau_p = stats.kendalltau(x, y)
    out.update(ols_slope=ols.slope, ols_intercept=ols.intercept,
               ols_r2=ols.rvalue ** 2, ols_p=ols.pvalue,
               ts_slope=ts[0], ts_intercept=ts[1], ts_lo=ts[2], ts_hi=ts[3],
               ts_tau=tau, ts_p=tau_p)
    return out


def plot_panel(ax, x, y, xlabel, ylabel, title, color_by):
    fit = fit_stats(x, y)
    sc = ax.scatter(x, y, c=color_by, cmap="viridis", s=30,
                    edgecolor="k", linewidth=0.3, zorder=3)
    if "ols_slope" in fit:
        xs = np.array([np.nanmin(x), np.nanmax(x)])
        ax.plot(xs, fit["ols_intercept"] + fit["ols_slope"] * xs,
                "-", color="#d62728", lw=1.7, label="OLS", zorder=2)
        ax.plot(xs, fit["ts_intercept"] + fit["ts_slope"] * xs,
                "--", color="#1f77b4", lw=1.7, label="Theil–Sen", zorder=2)
        txt = (f"OLS  m={fit['ols_slope']:+.4g}\n"
               f"     R²={fit['ols_r2']:.3f}  p={fit['ols_p']:.2g}\n"
               f"T–S  m={fit['ts_slope']:+.4g}\n"
               f"     τ={fit['ts_tau']:+.2f}  p={fit['ts_p']:.2g}\n"
               f"n = {fit['n']}")
        ax.legend(loc="lower left", fontsize=7, framealpha=0.75)
    else:
        txt = f"n = {fit['n']}\n(insufficient x-spread\nto fit)"
    ax.text(0.97, 0.97, txt, transform=ax.transAxes, ha="right", va="top",
            fontsize=8, family="monospace", zorder=4,
            bbox=dict(boxstyle="round", fc="white", ec="0.7", alpha=0.9))
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.grid(alpha=0.25)
    return fit, sc

## 5. The four panels

Row 1 plots reflectance and radiance side by side against **SZA**; row 2 plots the
same two against **cos(SZA)**. The shared colorbar encodes acquisition year.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9.5))

_, sc = plot_panel(axes[0, 0], df["sza"], df["rfl"],
                   "SZA (degrees)", RFL_LABEL, "Reflectance vs SZA", df["year"])
plot_panel(axes[0, 1], df["sza"], df["rad"],
           "SZA (degrees)", RAD_LABEL, "Radiance vs SZA", df["year"])
plot_panel(axes[1, 0], df["cos_sza"], df["rfl"],
           "cos(SZA)", RFL_LABEL, "Reflectance vs cos(SZA)", df["year"])
plot_panel(axes[1, 1], df["cos_sza"], df["rad"],
           "cos(SZA)", RAD_LABEL, "Radiance vs cos(SZA)", df["year"])

fig.suptitle(f"{PICS_ID}  patch {PATCH_ID}   —   "
             f"{WAVELENGTH_NM:.0f} nm ({Y_STAT} over patch, {len(df)} scenes)",
             fontsize=13, y=0.995)
fig.tight_layout(rect=(0, 0, 0.93, 0.98))
cax = fig.add_axes((0.945, 0.12, 0.015, 0.76))
fig.colorbar(sc, cax=cax, label="acquisition year")

FIG_DIR.mkdir(parents=True, exist_ok=True)
out_png = FIG_DIR / f"simple_linear_BRDF_{PICS_ID}_{PATCH_ID}_{WAVELENGTH_NM:.0f}nm.png"
fig.savefig(out_png, dpi=200, bbox_inches="tight")
print(f"Saved {out_png}")
plt.show()

## 6. Fit summary table

The BRDF-relevant number is the **cos(SZA)** slope: for reflectance, slope /
intercept is the site's fractional directional sensitivity ($d/c$). OLS and
Theil–Sen slopes are tabulated for all four panels.

In [ ]:
summary = []
for xcol, xname in [("sza", "SZA"), ("cos_sza", "cos(SZA)")]:
    for ycol, yname in [("rfl", "reflectance"), ("rad", "radiance")]:
        f = fit_stats(df[xcol], df[ycol])
        row = {"y": yname, "x": xname, "n": f["n"]}
        if "ols_slope" in f:
            row.update({"ols_slope": f["ols_slope"], "ols_r2": f["ols_r2"],
                        "ols_p": f["ols_p"], "ts_slope": f["ts_slope"],
                        "ts_p": f["ts_p"]})
            if xcol == "cos_sza" and f["ols_intercept"] != 0:
                row["ols_slope/intercept"] = f["ols_slope"] / f["ols_intercept"]
        summary.append(row)

pd.DataFrame(summary)